# Youtube Data API: Hilos Comentarios
https://developers.google.com/youtube/v3/docs/comments

https://developers.google.com/youtube/v3/docs/commentThreads

## Contenido:

* [Librerias](*Librerias)
* [Videos](*Videos)


<a name="Librerias"></a>
## Librerias

In [ ]:
# ========= Data Cleaning =========

!pip install advertools
!pip install clean-text
!pip install urlextract
import advertools as adv
from urlextract import URLExtract
from nltk.tokenize import regexp_tokenize
import html
import string 
import re
from cleantext import clean
import unicodedata

# ========= Data Cleaning =========

import pandas as pd
import requests
import csv
import time
from datetime import datetime
from datetime import date
import os.path
from os import path
import math
from google.colab import drive
key='TU_CLAVE_DE_API'
drive.mount('/content/drive/')

#### https://www.youtube.com/playlist?list=
#### https://www.youtube.com/channel/
#### https://www.youtube.com/watch?v=

## Hilos de Comentarios

### Ejecucion

#### Parametros constantes

In [ ]:
URIch = 'https://www.googleapis.com/youtube/v3/commentThreads?key='

# ================== Parametros Requeridos ==================

part = 'snippet,id' # Parametro obligatorio

# ================== Parametros Opcionales ==================

maxResults='50' # Numero maximo de elementos a devolver. El rango es 1 a 100, no se puede usar con el parametro id.
order = 'relevance' # Orden en que se deben devolver los comentarios, time = se ordenan por tiempo (valor predeterminado),
# relevance = se ordenan por relevancia, no se puede usar con el parametro id
# searchTerms = limite la respuesta de la API para que solo contenga comentarios que contengan los términos de búsqueda especificados,
# no se puede usar con el parametro id
textFormat = 'plainText' # Comentarios con formato HTML o como texto sin formato. html = comentarios con formato HTML (valor predeterminado), 
# plainText = comentarios en formato de texto sin formato

#### Parametros variables

In [ ]:
# ================== Filtros (especifique exactamente uno de los siguientes parámetros) ==================

# allThreadsRelatedToChannelId = Devuelve todos los hilos de comentarios asociados con el canal especificado. 
# La respuesta puede incluir comentarios sobre el canal o sobre los videos del canal.
# channelId = Devuelve hilos de comentarios que contengan comentarios sobre el canal especificado. 
# (La respuesta no incluirá los comentarios dejados en los videos que subió el canal).
# id = Lista separada por comas de ids de comentarios

videoId ='eHas3UkZe7g' # Devuelve hilos de comentarios asociados con el ID de video especificado

# ================== Parametros Opcionales ==================

pageToken = '' # Pagina especifica que se debe devolver, no se puede usar con el parametro id

#### Solicitud

In [ ]:
r = requests.get(URIch+key+'&part='+part+'&videoId='+videoId\
                 +'&maxResults='+maxResults+'&order='+order\
                 +'&pageToken='+pageToken+'&textFormat='+textFormat)

In [ ]:
r

In [ ]:
r.reason

In [ ]:
d = dict(r.json())

In [ ]:
d.keys()

In [ ]:
d

In [ ]:
d['kind']

In [ ]:
d['etag']

In [ ]:
d['nextPageToken']

In [ ]:
d['pageInfo']

#### Resultados

In [ ]:
items = d.get('items')
items

#### Dataframe

In [ ]:
data=pd.json_normalize(items)
data.info()

In [ ]:
data

### Algoritmo en Produccion

#### Importando lista de Videos 

In [ ]:
# ================== OJO modificar para cada serie ==================

Videos = pd.read_csv ('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/DataAnalisisVideos/\
Narcos/VR-VF/DataAnalisisVideosCompletos.csv')

# ================== OJO modificar para cada serie ==================

Videos

In [ ]:
Videos.info()

In [ ]:
Videos=Videos[Videos.ComentariosVideo.notnull()]
Videos.info()

In [ ]:
Videos[['ComentariosVideo']] = Videos[['ComentariosVideo']].astype(int)
Videos.info()

In [ ]:
Videos_mask=Videos['ComentariosVideo']!=0
filtered_Videos = Videos[Videos_mask]
filtered_Videos.info()

In [ ]:
dataVD=filtered_Videos.loc[:,['idVideo','TituloVideo','ComentariosVideo']]
dataVD

In [ ]:
dataVD['ComentariosVideo'].sum()

In [ ]:
ListaIDs=list(dataVD['idVideo'])
len(ListaIDs)

#### Parametros constantes

In [ ]:
URIch = 'https://www.googleapis.com/youtube/v3/commentThreads?key='

# ================== Parametros Requeridos ==================

part = 'snippet,id' # Parametro obligatorio
fields='pageInfo(*),nextPageToken(*),\
items(id,snippet(canReply,isPublic,totalReplyCount,videoId,channelId,\
topLevelComment(snippet(authorChannelId,authorChannelUrl,authorDisplayName,canRate,\
likeCount,publishedAt,textOriginal,updatedAt,viewerRating))))'

# ================== Parametros Opcionales ==================

maxResults = '100' # Numero maximo de elementos a devolver. El rango es 1 a 100, no se puede usar con el parametro id.
order = 'time' # Orden en que se deben devolver los comentarios, time = se ordenan por tiempo (valor predeterminado),
# relevance = se ordenan por relevancia, no se puede usar con el parametro id
# searchTerms = limite la respuesta de la API para que solo contenga comentarios que contengan los términos de búsqueda especificados,
# no se puede usar con el parametro id
textFormat = 'html' # Comentarios con formato HTML o como texto sin formato. html = comentarios con formato HTML (valor predeterminado), 
# plainText = comentarios en formato de texto sin formato

#### Parametros variables

In [ ]:
# ================== Filtros (especifique exactamente uno de los siguientes parámetros) ==================

ListIdVideo = ListaIDs # Devuelve hilos de comentarios asociados con el ID de video especificado

# ================== Parametros Opcionales ==================

pageToken = '' #pagina especifica que se debe devolver, no se puede usar con el parametro id

#### Carpetas Drive

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Comentarios/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'COMENTARIOS-Fecha'+str(datetime.now()) # Cambiar cuando sea por fecha reciente

#### Constantes

In [ ]:
solicitud = 0 # Inicio de las solicitudes
espera = 0.5 # Valor de retardo para las solicitudes
solicitudmax = 6000 # Numero de solicitudes maximas (el limite es 10000)

# ======================= OJO Cambiar diariamente =======================

ithid=0 # i-esimo id

# ======================= OJO Cambiar diariamente =======================

#### Ejecucion

In [ ]:
Datac= pd.DataFrame() # Dataframe de Resultados Totales

Metadatac = pd.DataFrame(columns=['Solicitud','idVideo','maxResultados','FechaEjecucion','Respuesta'\
                                  ,'prevPageToken','nextPageToken','ResultadosTotal','Resultadosxpagina']) # Dataframe de Informacion de Solicitudes

Faltantes=pd.DataFrame(columns=['idVideo','Solicitud'])

for idvideo in ListIdVideo[ithid:]: # For de id's de videos

  if (solicitud == solicitudmax+1):
    break

  filtro=dataVD[dataVD['idVideo'] == idvideo]
  Comentarios = list(filtro['ComentariosVideo'])[0]
  paginasmax = math.ceil(Comentarios/100) # Numero de paginas maxinas para los comentarios
  pageToken = '' # Pagina especifica que se debe devolver OJO DEBE DE IR EN ESTE LUGAR

  for pagina in range(1,paginasmax+1): # For de la paginacion

    solicitud = solicitud+1
    
    if (solicitud == solicitudmax+1):
      ptoPartida = [idvideo, pagina] # Punto de partida para volver a comenzar el motor de busqueda en produccion.
      break

    time.sleep(espera) # Tiempo de espera por cada solicitud
    r = requests.get(URIch+key+'&part='+part+'&fields='+fields+'&videoId='+idvideo\
                     +'&maxResults='+maxResults+'&order='+order\
                     +'&pageToken='+pageToken+'&textFormat='+textFormat) # Solicitud
    
    print('solicitud',solicitud,r.reason)

    if (r.reason=='OK'): # Vemos si la solicitud fue exitosa

      d = dict(r.json())
      claves=list(d.keys())
      items = d.get('items')
      Data=pd.json_normalize(items).assign(Solicitud=solicitud)
      Datac=pd.concat([Datac, Data], ignore_index=True)

      if ('nextPageToken' in claves): # Si hay siguiente pagina
        pageToken=d['nextPageToken'] # Definimos el proximo pagetoken
        if ('prevPageToken' in claves):
          prevPageToken=d['prevPageToken']
        else:
          prevPageToken=''

        Metadatac = Metadatac.append({'Solicitud':solicitud,'idVideo':idvideo,'maxResultados':maxResults\
                                      ,'FechaEjecucion':str(datetime.now()),'Respuesta':r.reason\
                                      ,'prevPageToken':prevPageToken,'nextPageToken':d['nextPageToken']\
                                      ,'ResultadosTotal':d['pageInfo']['totalResults'],'Resultadosxpagina':d['pageInfo']['resultsPerPage']}, ignore_index=True)
        print("Siguiente pagina")

      else: # Si no hay siguiente pagina
        nextPageToken='' # Definimos el nextPageToken como vacio
        if ('prevPageToken' in claves):
          prevPageToken=d['prevPageToken']
        else:
          prevPageToken=''
        
        Metadatac = Metadatac.append({'Solicitud':solicitud,'idVideo':idvideo,'maxResultados':maxResults\
                                      ,'FechaEjecucion':str(datetime.now()),'Respuesta':r.reason\
                                      ,'prevPageToken':prevPageToken,'nextPageToken':nextPageToken\
                                      ,'ResultadosTotal':d['pageInfo']['totalResults'],'Resultadosxpagina':d['pageInfo']['resultsPerPage']}, ignore_index=True)
        print("No hay mas paginas")
        break

    else: # Vemos si la solicitud no fue exitosa
      Faltantes=Faltantes.append({'idVideo':idvideo,'Solicitud':solicitud}, ignore_index=True)
      print("La solicitud no se pudo realizar para el video ",idvideo)
      break

print("Terminados los comentarios")

In [ ]:
ptoPartida

In [ ]:
Datac.info()

In [ ]:
Datac

In [ ]:
Metadatac.info()

In [ ]:
Metadatac

In [ ]:
Faltantes

#### Estructurar y dar formato a Dataframe

In [ ]:
columnas={'id':'idComentarioPrincipal','snippet.videoId':'idVideo','snippet.topLevelComment.snippet.textOriginal':'TextoComentarioPrincipal'\
          ,'snippet.topLevelComment.snippet.authorDisplayName':'NombreAutorComentarioPrincipal'\
          ,'snippet.topLevelComment.snippet.authorChannelUrl':'URLAutorCanalComentarioPrincipal','snippet.topLevelComment.snippet.authorChannelId.value':'idAutorCanalComentarioPrincipal'\
          ,'snippet.topLevelComment.snippet.canRate':'CalificarComentarioPrincipal','snippet.topLevelComment.snippet.viewerRating':'CalificacionEspectadorComentarioPrincipal'\
          ,'snippet.topLevelComment.snippet.likeCount':'LikesComentarioPrincipal','snippet.topLevelComment.snippet.publishedAt':'FechaPublicacionComentarioPrincipal'\
          ,'snippet.topLevelComment.snippet.updatedAt':'FechaActualizacionComentarioPrincipal','snippet.canReply':'ReplicarComentarioPrincipal'\
          ,'snippet.totalReplyCount':'ReplicasTotalesComentarioPrincipal','snippet.isPublic':'PublicoComentarioPrincipal','Solicitud':'SolicitudComentarioPrincipal'}

columnasorden=['idVideo','idComentarioPrincipal','TextoComentarioPrincipal','LikesComentarioPrincipal','ReplicasTotalesComentarioPrincipal','FechaPublicacionComentarioPrincipal','FechaActualizacionComentarioPrincipal'\
               ,'PublicoComentarioPrincipal','ReplicarComentarioPrincipal','NombreAutorComentarioPrincipal','URLAutorCanalComentarioPrincipal','idAutorCanalComentarioPrincipal','CalificacionEspectadorComentarioPrincipal','CalificarComentarioPrincipal','SolicitudComentarioPrincipal']

Datac.rename(columns=columnas,inplace=True)
DATA=Datac.reindex(columns=columnasorden)
DATA

In [ ]:
DATA.info()

#### Extraccion de datos

Se generan 4 nuevas columnas (emoticones, hashtags, URLs y menciones) por cada columna de texto importante. (En este caso 4)

##### Extreaer emoticones

In [ ]:
def emojis(texto):
  if (len(texto) != 0):
    emoji_summary = adv.extract_emoji(texto)
    return emoji_summary['top_emoji']
  else:
    L=[]
    return L

In [ ]:
DATA['emojisComentarioPrincipal']=DATA['TextoComentarioPrincipal'].apply(emojis)

In [ ]:
DATA['emojisComentarioPrincipal']

##### Extraer URLs

In [ ]:
extractor = URLExtract()
def URLs(texto):
  URL=extractor.find_urls(texto)
  return URL

In [ ]:
DATA['URLsComentarioPrincipal']=DATA['TextoComentarioPrincipal'].apply(URLs)

In [ ]:
DATA['URLsComentarioPrincipal']

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['TextoComentarioPrincipal'].apply(codigohex)

##### Extraer Hastags

In [ ]:
def hastags(texto):
  pattern='#\w+'
  hastag=regexp_tokenize(texto,pattern)
  return hastag

In [ ]:
DATA['hashtagsComentarioPrincipal']=DATA['ComentarioPrincipalModificado'].apply(hastags)

In [ ]:
DATA['hashtagsComentarioPrincipal']

##### Extraer menciones o etiquetas (puede ser necesario alguna modificacion para el caso de correos)

In [ ]:
def mencion(texto):
  pattern='@\w+'
  mencion=regexp_tokenize(texto,pattern)
  return mencion

In [ ]:
DATA['MencionesComentarioPrincipal']=DATA['TextoComentarioPrincipal'].apply(mencion)

In [ ]:
DATA['MencionesComentarioPrincipal']

#### DATA

In [ ]:
DATA.info()

#### Limpiar texto: (Generamos una copia llamada columname_modificado de una columna a partir del cual vamos a realizar la limpieza de datos) OJO: Revisar secuencias de escape para serie tres caines y los 80

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['TextoComentarioPrincipal'].apply(codigohex)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar Unicode (Es aca donde se eliminan las tildes-> Solucionado) NOTA: No se entiende muy bien que realiza este parte del codigo

In [ ]:
def dropunicode(texto):
  texto = unicodedata.normalize('NFD', texto)
  texto=texto.encode('ascii','ignore').decode('utf8')
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropunicode)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar secuencias de escape

In [ ]:
DATA['TextoComentarioPrincipal'] = DATA['TextoComentarioPrincipal'].apply(str.replace, args=('\r', ' '))
DATA['TextoComentarioPrincipal'] = DATA['TextoComentarioPrincipal'].apply(str.replace, args=('\n', ' '))
DATA['TextoComentarioPrincipal'] = DATA['TextoComentarioPrincipal'].apply(str.replace, args=('\v', ' '))
DATA['TextoComentarioPrincipal'] = DATA['TextoComentarioPrincipal'].apply(str.replace, args=('\t', ' '))
DATA['TextoComentarioPrincipal'] = DATA['TextoComentarioPrincipal'].apply(str.replace, args=('\f', ' '))
DATA['TextoComentarioPrincipal'] = DATA['TextoComentarioPrincipal'].apply(str.replace, args=('\b', ' '))
#DATA['TextoComentarioPrincipal'] = DATA['TextoComentarioPrincipal'].apply(str.replace, args=('3', 'tres')) #serie tres caines
#DATA['TextoComentarioPrincipal'] = DATA['TextoComentarioPrincipal'].apply(str.replace, args=('80', 'ochenta')) # serie los 80
DATA['ComentarioPrincipalModificado'] = DATA['ComentarioPrincipalModificado'].apply(str.replace, args=('\r', ' '))
DATA['ComentarioPrincipalModificado'] = DATA['ComentarioPrincipalModificado'].apply(str.replace, args=('\n', ' '))
DATA['ComentarioPrincipalModificado'] = DATA['ComentarioPrincipalModificado'].apply(str.replace, args=('\v', ' '))
DATA['ComentarioPrincipalModificado'] = DATA['ComentarioPrincipalModificado'].apply(str.replace, args=('\t', ' '))
DATA['ComentarioPrincipalModificado'] = DATA['ComentarioPrincipalModificado'].apply(str.replace, args=('\f', ' '))
DATA['ComentarioPrincipalModificado'] = DATA['ComentarioPrincipalModificado'].apply(str.replace, args=('\b', ' '))
#DATA['ComentarioPrincipalModificado'] = DATA['ComentarioPrincipalModificado'].apply(str.replace, args=('3', 'tres')) # serie tres caines
#DATA['ComentarioPrincipalModificado'] = DATA['ComentarioPrincipalModificado'].apply(str.replace, args=('80', 'ochenta')) # serie los ochenta

##### Eliminar URLs

In [ ]:
def dropURLs(texto):
  text = re.sub('https?://\S+|www\.\S+', ' ', texto)
  return text
def dropURL(texto):
  text = re.sub('http?://\S+|www\.\S+', ' ', texto)
  return text

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropURLs)
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropURL)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar Emoticones

In [ ]:
from cleantext import clean

In [ ]:
from dataclasses import replace
def dropemoticons(texto):
  texto=clean(texto, no_emoji=True,lower=False)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropemoticons)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar emails

In [ ]:
def dropemails(texto):
  texto=clean(texto, no_emails=True, lower=False)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropemails)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar Menciones

In [ ]:
def dropmenciones(texto):
  texto = re.sub("@\S+", " ", texto)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropmenciones)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Quitar caracteres repetidos


In [ ]:
def droppalabrasrepetidas(texto):
  texto = re.sub(r'(.)\1{3,}',r'\1', texto)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(droppalabrasrepetidas)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar Numeros

In [ ]:
def dropnumeros(texto):
  pattern = r'[0-9]'
  texto = re.sub(pattern, '', texto)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropnumeros)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar \# de Hashtags

In [ ]:
def dropnumeral(texto):
  texto = re.sub(r'#', ' ', texto)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropnumeral)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Dividir palabras adjuntas (CamelCase)

In [ ]:
def splitpalabras(texto):
  texto = " ".join([s for s in re.split("([A-Z][a-z]+[^A-Z]*)",texto) if s])
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(splitpalabras)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar signos de puntuacion

In [ ]:
punct = set(string.punctuation) 
punct 

In [ ]:
def puntuacion(texto):
  sinpuntuacion="".join([ch for ch in texto if ch not in punct])
  return sinpuntuacion

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(puntuacion)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar espacios

In [ ]:
def dropespacios(texto):
  texto = re.sub('\s{2,}', " ", texto)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropespacios)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminar tokens cortos

In [ ]:
def droppalabrascortas(texto):
  tokens = texto.split()
  clean_tokens = [t for t in tokens if len(t) > 1]
  clean_text = " ".join(clean_tokens)
  return clean_text

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(droppalabrascortas)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Normalizar texto (Quitar mayusculas)

In [ ]:
def normatext(texto):
  textonormalizado=texto.lower()
  return textonormalizado

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(normatext)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Cambios particulares para cada serie

In [ ]:
def cambiosEPM(texto):
  texto1=re.sub("elpatrondelmal", "el patron del mal", texto)
  texto2=re.sub("patrondelmal", "patron del mal", texto1)
  texto3=re.sub("patrondel", "patron del", texto2)
  texto4=re.sub("pabloescobar", "pablo escobar", texto3)
  texto5=re.sub("murtes", "muertes", texto4)
  texto6=re.sub("murte", "muerte", texto5)
  texto7=re.sub("muerts", "muertes", texto6)
  texto8=re.sub("patron", " patron ", texto7)
  texto9=re.sub("escobar", " escobar ", texto8)
  texto=re.sub("pablo", " pablo ", texto9)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(cambiosEPM)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Eliminando stopwords

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
swsp=nltk.corpus.stopwords.words('spanish')

In [ ]:
swpo=nltk.corpus.stopwords.words('portuguese')

In [ ]:
swen=nltk.corpus.stopwords.words('english')

In [ ]:
def dropStopWordsSpanish(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swpo:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
def dropStopWordsPortugues(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swsp:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
def dropStopWordsEnglish(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swen:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropStopWordsSpanish)
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropStopWordsPortugues)
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropStopWordsEnglish)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Volvemos a eliminar exceso de espacios

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(dropespacios)

In [ ]:
DATA['ComentarioPrincipalModificado']

##### Volvemos a eliminar tokens cortos

In [ ]:
DATA['ComentarioPrincipalModificado']=DATA['ComentarioPrincipalModificado'].apply(droppalabrascortas)

In [ ]:
DATA['ComentarioPrincipalModificado']

#### Estructurar y dar formato

In [ ]:
DATA.info()

In [ ]:
columnasorden=['idVideo','idComentarioPrincipal','TextoComentarioPrincipal','ComentarioPrincipalModificado','emojisComentarioPrincipal','URLsComentarioPrincipal','hashtagsComentarioPrincipal','MencionesComentarioPrincipal'\
               ,'LikesComentarioPrincipal','ReplicasTotalesComentarioPrincipal','FechaPublicacionComentarioPrincipal','FechaActualizacionComentarioPrincipal'\
               ,'PublicoComentarioPrincipal','ReplicarComentarioPrincipal','NombreAutorComentarioPrincipal','URLAutorCanalComentarioPrincipal','idAutorCanalComentarioPrincipal','CalificacionEspectadorComentarioPrincipal','CalificarComentarioPrincipal','SolicitudComentarioPrincipal']

DATA=DATA.reindex(columns=columnasorden)
DATA

In [ ]:
DATA.info()

#### Exportar Dataframe a CSV

In [ ]:
Metadatac.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)
Faltantes.to_csv(PATH_TO_DIR_0+'/'+'faltantes.csv', index=False)
DATA.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)

## Concatenando bases de datos de comentarios relevantes y por fecha reciente, ademas eliminamos duplicados

In [ ]:
# ================== OJO modificar para cada serie ==================

ComentariosR = pd.read_csv ('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Comentarios/\
Los80,MasqueUnaModa/2022-09-07/RESULTS-COMENTARIOS-2022-09-07 18:06:42.690616.csv')

ComentariosF = pd.read_csv ('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Comentarios/\
Los80,MasqueUnaModa/2022-09-07/RESULTS-COMENTARIOS-Fecha2022-09-07 18:20:40.630640.csv')

# ================== OJO modificar para cada serie ==================


In [ ]:
ComentariosR

In [ ]:
ComentariosF

In [ ]:
DATA=pd.concat([ComentariosR,ComentariosF])

In [ ]:
DATA

In [ ]:
DATAclear=DATA.drop_duplicates(subset=['idComentarioPrincipal'])
DATAclear.info()

In [ ]:
DATAclear

In [ ]:
DATAclear=DATAclear.sort_values('idVideo').reset_index().drop(columns=['index'])
DATAclear

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Comentarios/Los80,MasqueUnaModa/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today +'CR(2022-09-07)'+'CF(2022-09-07)'
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'COMENTARIOS'+str(datetime.now()) # +'Relevance' ========> OJO modificar si es por relevancia o por numero de vistas

In [ ]:
DATAclear.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)